In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_percentage_error
from keras.models import Sequential
from keras.layers import GRU,Dense,Dropout
from keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv("rnn_sequence_dataset.csv")

In [3]:
df

,t,value
0,0.000000,0.034092
1,0.125916,0.272815
2,0.251831,0.120151
3,0.377747,0.326518
4,0.503662,0.401622
...,...,...
495,62.328191,-0.483172
496,62.454106,-0.292410
497,62.580022,-0.225605
498,62.705938,-0.227046


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   t       500 non-null    float64
 1   value   500 non-null    float64
dtypes: float64(2)
memory usage: 7.9 KB


In [5]:
scaler=MinMaxScaler()
df["value"]=scaler.fit_transform(df[["value"]])

In [6]:
def create_dataset(data, window=20):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i:i+window])
        y.append(data[i+window])
    return np.array(X), np.array(y)



X,y=create_dataset(df["value"])


In [7]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=False)


In [8]:
model=Sequential([
    GRU(64,input_shape=(20,1),return_sequences=True),
    GRU(32),
    Dense(1)
])

In [9]:
model.compile(optimizer="adam",loss="mse")

In [10]:
history=model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test,),callbacks=[EarlyStopping(monitor="val_loss",patience=3)])

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 17s 478ms/step - loss: 0.1417 - val_loss: 0.0389
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 7s 100ms/step - loss: 0.0404 - val_loss: 0.0223
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 135ms/step - loss: 0.0244 - val_loss: 0.0186
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step - loss: 0.0164 - val_loss: 0.0143
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 94ms/step - loss: 0.0133 - val_loss: 0.0104
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 82ms/step - loss: 0.0105 - val_loss: 0.0079
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - loss: 0.0082 - val_loss: 0.0057
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 0.0060 - val_loss: 0.0036
Epoch 9/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 74ms/step - loss: 0.0042 - val_loss: 0.0022
Epoch 10/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step - loss: 0.0032 - val_loss: 0.0018
Epoch 11/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.0031 - val_loss: 0.0018
Epoch 12/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/

In [11]:
y_perd=model.predict(X_test)
y_pred=scaler.inverse_transform(y_perd)
y_test=scaler.inverse_transform(y_test.reshape(-1,1))

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step 


In [12]:
mean_absolute_percentage_error(y_test,y_pred)

0.31365143871857243

In [13]:
r2_score(y_test,y_pred)


0.980736007908827